In [2]:
%pip install torchvision

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 6.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [135]:
import torch
from torch import nn
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import os
from torchvision import datasets, transforms

In [4]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100.0%
100.0%
100.0%
100.0%


In [5]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)


CrossEntropyLoss()

In [9]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [10]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [11]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.305272  [   64/60000]
loss: 2.291373  [ 6464/60000]
loss: 2.271004  [12864/60000]
loss: 2.265693  [19264/60000]
loss: 2.243301  [25664/60000]
loss: 2.212319  [32064/60000]
loss: 2.225577  [38464/60000]
loss: 2.189852  [44864/60000]
loss: 2.185015  [51264/60000]
loss: 2.150664  [57664/60000]
Test Error: 
 Accuracy: 43.6%, Avg loss: 2.148536 

Epoch 2
-------------------------------
loss: 2.159120  [   64/60000]
loss: 2.152195  [ 6464/60000]
loss: 2.086610  [12864/60000]
loss: 2.106568  [19264/60000]
loss: 2.068517  [25664/60000]
loss: 1.996971  [32064/60000]
loss: 2.032383  [38464/60000]
loss: 1.950067  [44864/60000]
loss: 1.948103  [51264/60000]
loss: 1.889355  [57664/60000]
Test Error: 
 Accuracy: 60.7%, Avg loss: 1.883353 

Epoch 3
-------------------------------
loss: 1.911600  [   64/60000]
loss: 1.886822  [ 6464/60000]
loss: 1.759018  [12864/60000]
loss: 1.806414  [19264/60000]
loss: 1.714883  [25664/60000]
loss: 1.652127  [32064/600

In [16]:
data = [[1, 2, 3], [4, 5, 6]]
x_data = torch.tensor(data)

x_data

tensor([[1, 2, 3],
        [4, 5, 6]])

In [20]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

np_array

array([[1, 2, 3],
       [4, 5, 6]])

In [25]:
x_ones = torch.ones_like(x_data)
print(f"One Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float)
print(f"Rand Tensor: \n {x_rand} \n")

One Tensor: 
 tensor([[1, 1, 1],
        [1, 1, 1]]) 

Rand Tensor: 
 tensor([[0.4509, 0.5107, 0.4980],
        [0.7801, 0.4565, 0.1695]]) 



In [26]:
shape = (30, 40)

rand_tensor = torch.rand(shape)
one_tensor = torch.ones(shape)
zero_tensor = torch.zeros(shape)

print(f"Random Tensor \n {rand_tensor} \n")
print(f"One Tensor \n {one_tensor} \n")
print(f"Zero Tensor \n {zero_tensor} \n")

Random Tensor 
 tensor([[0.4861, 0.0046, 0.2647,  ..., 0.3399, 0.7020, 0.5758],
        [0.8548, 0.6825, 0.3028,  ..., 0.1093, 0.4480, 0.6064],
        [0.7433, 0.1260, 0.4020,  ..., 0.8031, 0.3397, 0.8541],
        ...,
        [0.4538, 0.5229, 0.3899,  ..., 0.2009, 0.3237, 0.4922],
        [0.3787, 0.6617, 0.9204,  ..., 0.1780, 0.7073, 0.0702],
        [0.7454, 0.3194, 0.3107,  ..., 0.1645, 0.0221, 0.5018]]) 

One Tensor 
 tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]]) 

Zero Tensor 
 tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]) 



In [28]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")
print(tensor)

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu
tensor([[0.3215, 0.1549, 0.1295, 0.2893],
        [0.3787, 0.7099, 0.8279, 0.0648],
        [0.5183, 0.0421, 0.0975, 0.5893]])


In [29]:
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

In [37]:
tensor = torch.ones(4, 4)
print(f"First Row: {tensor[0]}")
print(f"First Col: {tensor[:,0]}")
print(f"Last Col: {tensor[:, -1]}")
tensor[:, 1] = 0
print(tensor)
tensor[:, 1] = torch.rand(len(tensor))
print(tensor)

First Row: tensor([1., 1., 1., 1.])
First Col: tensor([1., 1., 1., 1.])
Last Col: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])
tensor([[1.0000, 0.1851, 1.0000, 1.0000],
        [1.0000, 0.1871, 1.0000, 1.0000],
        [1.0000, 0.3050, 1.0000, 1.0000],
        [1.0000, 0.0540, 1.0000, 1.0000]])


In [39]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)
t1.shape

tensor([[1.0000, 0.1851, 1.0000, 1.0000, 1.0000, 0.1851, 1.0000, 1.0000, 1.0000,
         0.1851, 1.0000, 1.0000],
        [1.0000, 0.1871, 1.0000, 1.0000, 1.0000, 0.1871, 1.0000, 1.0000, 1.0000,
         0.1871, 1.0000, 1.0000],
        [1.0000, 0.3050, 1.0000, 1.0000, 1.0000, 0.3050, 1.0000, 1.0000, 1.0000,
         0.3050, 1.0000, 1.0000],
        [1.0000, 0.0540, 1.0000, 1.0000, 1.0000, 0.0540, 1.0000, 1.0000, 1.0000,
         0.0540, 1.0000, 1.0000]])


torch.Size([4, 12])

In [41]:
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)

z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)

torch.mul(tensor, tensor, out=z3)

tensor([[1.0000, 0.0343, 1.0000, 1.0000],
        [1.0000, 0.0350, 1.0000, 1.0000],
        [1.0000, 0.0930, 1.0000, 1.0000],
        [1.0000, 0.0029, 1.0000, 1.0000]])

In [42]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

12.731244087219238 <class 'float'>


In [43]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

tensor([[1.0000, 0.1851, 1.0000, 1.0000],
        [1.0000, 0.1871, 1.0000, 1.0000],
        [1.0000, 0.3050, 1.0000, 1.0000],
        [1.0000, 0.0540, 1.0000, 1.0000]]) 

tensor([[6.0000, 5.1851, 6.0000, 6.0000],
        [6.0000, 5.1871, 6.0000, 6.0000],
        [6.0000, 5.3050, 6.0000, 6.0000],
        [6.0000, 5.0540, 6.0000, 6.0000]])


In [44]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


In [45]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


In [105]:
shape = (3,4)
zero = torch.zeros(shape)
zero[:, -1] = torch.rand(len(zero))
a = torch.ones(2, 3)
b = a @ a.T
c = a * a

x = torch.arange(6)
x = x.reshape(2, 3)
print(f"Zero: {zero}")
print(f"X: {x}")
print(f"B: {b}")
print(f"C: {c}")

print(f"Row: {x[1]}")
print(f"Col: 1.{x.T[0]}")
print(f"Col: 2.{x[:, 0]}")

t = x.numpy()
print(f"T: {t}")

x.add_(10)
print(f"X: {x}")
print(f"T: {t}")

Zero: tensor([[0.0000, 0.0000, 0.0000, 0.5132],
        [0.0000, 0.0000, 0.0000, 0.1149],
        [0.0000, 0.0000, 0.0000, 0.3195]])
X: tensor([[0, 1, 2],
        [3, 4, 5]])
B: tensor([[3., 3.],
        [3., 3.]])
C: tensor([[1., 1., 1.],
        [1., 1., 1.]])
Row: tensor([3, 4, 5])
Col: 1.tensor([0, 3])
Col: 2.tensor([0, 3])
T: [[0 1 2]
 [3 4 5]]
X: tensor([[10, 11, 12],
        [13, 14, 15]])
T: [[10 11 12]
 [13 14 15]]


In [109]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [110]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [111]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [112]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")


Predicted class: tensor([4])


In [113]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [115]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [118]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [120]:
print(f"Before ReLU: {hidden1}")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[0.0000, 0.3505, 0.0361, 0.0814, 0.0000, 0.0000, 0.0000, 0.0000, 0.4690,
         0.3214, 0.0000, 0.4790, 0.5871, 0.0744, 0.4554, 0.6534, 0.0000, 0.0000,
         0.6714, 0.0000],
        [0.0000, 0.2006, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0849, 0.0000,
         0.3112, 0.0000, 0.0502, 0.6190, 0.0000, 0.3290, 0.5132, 0.0000, 0.0000,
         0.9004, 0.0000],
        [0.0000, 0.3159, 0.3287, 0.0250, 0.0000, 0.0000, 0.0000, 0.1787, 0.3375,
         0.4452, 0.0000, 0.4726, 0.5033, 0.3784, 0.0178, 0.3485, 0.0000, 0.0000,
         0.3114, 0.0000]], grad_fn=<ReluBackward0>)
After ReLU: tensor([[0.0000, 0.3505, 0.0361, 0.0814, 0.0000, 0.0000, 0.0000, 0.0000, 0.4690,
         0.3214, 0.0000, 0.4790, 0.5871, 0.0744, 0.4554, 0.6534, 0.0000, 0.0000,
         0.6714, 0.0000],
        [0.0000, 0.2006, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0849, 0.0000,
         0.3112, 0.0000, 0.0502, 0.6190, 0.0000, 0.3290, 0.5132, 0.0000, 0.0000,
         0.9004, 0.0000],
        [

In [125]:
seq_modules = nn.Sequential(
    flatten,
    layer1, 
    nn.ReLU(), 
    nn.Linear(20,10)
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)


In [126]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [ ]:
print(f"Model Structure: {model} \n\n")

for param, names in model.named_parameters():
    nn.Linear()
    print(f"Layers: {names} | Paramaters: {param} | Value: {param[:2]} \n")

Model Structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
) 


Layers: Parameter containing:
tensor([[ 0.0339, -0.0092,  0.0233,  ...,  0.0332,  0.0228, -0.0139],
        [ 0.0044, -0.0056, -0.0138,  ...,  0.0162, -0.0024,  0.0242],
        [ 0.0271,  0.0292,  0.0252,  ..., -0.0356,  0.0214,  0.0025],
        ...,
        [ 0.0178,  0.0156, -0.0178,  ..., -0.0077,  0.0225,  0.0254],
        [-0.0132,  0.0314,  0.0075,  ..., -0.0012, -0.0179, -0.0103],
        [ 0.0195,  0.0314,  0.0287,  ...,  0.0139, -0.0274,  0.0337]],
       requires_grad=True) | Paramaters: linear_relu_stack.0.weight | Value: li 

Layers: Parameter containing:
tensor([ 5.9121e-03, -2.0688e-02,  2.3485e-02,  3.4689e-02, -2.0416e-02,
        -2.6

In [137]:
x = torch.ones(5)
y = torch.zeros(3)
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z,y)

print(f"X or Input Tensor: {x}")
print(f"Y or Expected outputs: {y}")
print(f"W or Weights: {w}")
print(f"B or Biases: {b}")
print(f"Z or Pre-Activation: {z}")
print(f"Loss or Cost: {loss}")

X or Input Tensor: tensor([1., 1., 1., 1., 1.])
Y or Expected outputs: tensor([0., 0., 0.])
W or Weights: tensor([[-0.0379, -0.2519,  0.1513],
        [ 0.6416, -0.8155, -0.4621],
        [ 0.4215, -1.0042,  0.2715],
        [ 0.4453,  2.0312,  0.1977],
        [-0.8283, -1.0053,  0.8355]], requires_grad=True)
B or Biases: tensor([ 1.1621, -0.9800, -0.2152], requires_grad=True)
Z or Pre-Activation: tensor([ 1.8045, -2.0257,  0.7785], grad_fn=<AddBackward0>)
Loss or Cost: 1.079013466835022


In [138]:
print(f"Gradient for function z = {z.grad_fn}")
print(f"Gradient for loss funcition = {loss.grad_fn}")

Gradient for function z = <AddBackward0 object at 0x0000021400072470>
Gradient for loss funcition = <BinaryCrossEntropyWithLogitsBackward0 object at 0x00000214014075B0>


In [139]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2862, 0.0388, 0.2285],
        [0.2862, 0.0388, 0.2285],
        [0.2862, 0.0388, 0.2285],
        [0.2862, 0.0388, 0.2285],
        [0.2862, 0.0388, 0.2285]])
tensor([0.2862, 0.0388, 0.2285])


In [141]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
    print(z.requires_grad) 

True
False


In [144]:
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z.requires_grad)
print(z_det.requires_grad)

True
False


In [146]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp + 1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call \n {inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call 
 tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data"
    train=False
    download=True
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)